# Titanic Dataset — Data Preprocessing for Machine Learning

This notebook performs complete data preprocessing on the Titanic dataset:
loading & exploration, cleaning, feature engineering, encoding/scaling,
train/test split, a reusable Scikit-learn preprocessing pipeline, and
validation of the final output.


## 1. Load & Explore Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

df = pd.read_csv('data/titanic.csv')
print("Shape:", df.shape)
df.head()


Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print("Columns:", list(df.columns))
print()
df.info()


Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [3]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Braund, Mr. Owen Harris",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


In [4]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)
print(missing_summary)


          missing_count  missing_pct
Cabin               687        77.10
Age                 177        19.87
Embarked              2         0.22


In [5]:
# Duplicate records
n_dupes = df.duplicated().shape[0] - df.drop_duplicates().shape[0]
print(f"Duplicate rows: {n_dupes}")


Duplicate rows: 0


In [6]:
# Target distribution
print(df['Survived'].value_counts())
print(df['Survived'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(4,3))
df['Survived'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#d62728', '#2ca02c'])
ax.set_xticklabels(['Died (0)', 'Survived (1)'], rotation=0)
ax.set_title('Target distribution: Survived')
plt.tight_layout()
plt.savefig('outputs/target_distribution.png', dpi=100)
plt.show()


Survived
0    549
1    342
Name: count, dtype: int64
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64


## 2. Data Cleaning

Steps:
- Drop exact duplicate rows.
- Impute `Age` and `Fare` with the **median** (robust to outliers/skew).
- Impute `Embarked` with the **mode** (only 2 missing values).
- Analyze `Cabin` (77% missing) — convert to a binary `HasCabin` indicator
  instead of dropping outright, since cabin *presence* correlates with
  passenger class/fare and carries signal even though the raw value is
  too sparse to use directly.
- Drop `PassengerId`, `Ticket`, and raw `Name`/`Cabin` (irrelevant or
  too high-cardinality for direct modeling) after extracting what's
  useful from them (Title from Name, HasCabin from Cabin).


In [7]:
df_clean = df.copy()

# 2.1 Remove duplicate records
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after = df_clean.shape[0]
print(f"Removed {before - after} duplicate rows")


Removed 0 duplicate rows


In [8]:
# 2.2 Handle missing values
age_median = df_clean['Age'].median()
fare_median = df_clean['Fare'].median()
embarked_mode = df_clean['Embarked'].mode()[0]

print(f"Age median used for imputation: {age_median}")
print(f"Fare median used for imputation: {fare_median}")
print(f"Embarked mode used for imputation: {embarked_mode}")

df_clean['Age'] = df_clean['Age'].fillna(age_median)
df_clean['Fare'] = df_clean['Fare'].fillna(fare_median)
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)


Age median used for imputation: 28.0
Fare median used for imputation: 14.4542
Embarked mode used for imputation: S


In [9]:
# 2.3 Analyze and handle Cabin
cabin_missing_pct = df_clean['Cabin'].isnull().mean() * 100
print(f"Cabin missing: {cabin_missing_pct:.1f}% -> too sparse to impute meaningfully")

# Binary indicator: did the passenger have a recorded cabin?
df_clean['HasCabin'] = df_clean['Cabin'].notnull().astype(int)

# Sanity check: HasCabin vs survival rate
print(df_clean.groupby('HasCabin')['Survived'].mean())


Cabin missing: 77.1% -> too sparse to impute meaningfully
HasCabin
0    0.299854
1    0.666667
Name: Survived, dtype: float64


In [10]:
# 2.4 Remove irrelevant / high-cardinality identifier columns
# (Name and Cabin are kept one extra step for feature engineering below,
#  then dropped at the end of section 3.)
df_clean = df_clean.drop(columns=['PassengerId', 'Ticket'])
print("Remaining columns:", list(df_clean.columns))


Remaining columns: ['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'HasCabin']


## 3. Feature Engineering

New features:
- `FamilySize` = `SibSp` + `Parch` + 1 (includes the passenger themself)
- `IsAlone` = 1 if `FamilySize` == 1, else 0
- `Title` extracted from `Name` (Mr, Mrs, Miss, Master, and a grouped
  `Rare` bucket for uncommon titles), then `Name` and `Cabin` are dropped.


In [11]:
# 3.1 FamilySize
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

# 3.2 IsAlone
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

print(df_clean[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head())


   SibSp  Parch  FamilySize  IsAlone
0      1      0           2        0
1      1      0           2        0
2      0      0           1        1
3      1      0           2        0
4      0      0           1        1


In [12]:
# 3.3 Extract Title from Name
df_clean['Title'] = df_clean['Name'].str.extract(r',\s*([^\.]*)\.')
print(df_clean['Title'].value_counts())


Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [13]:
# Group rare / equivalent titles
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}
df_clean['Title'] = df_clean['Title'].replace(title_map)
# Anything not in the common set becomes 'Rare'
common_titles = {'Mr', 'Mrs', 'Miss', 'Master'}
df_clean['Title'] = df_clean['Title'].apply(lambda t: t if t in common_titles else 'Rare')
print(df_clean['Title'].value_counts())


Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


In [14]:
# Now drop Name and Cabin (raw text / too sparse — replaced by Title & HasCabin)
df_clean = df_clean.drop(columns=['Name', 'Cabin'])
df_clean.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,FamilySize,IsAlone,Title
0,0,3,male,22.0,1,0,7.2500,S,0,2,0,Mr
1,1,1,female,38.0,1,0,71.2833,C,1,2,0,Mrs
2,1,3,female,26.0,0,0,7.9250,S,0,1,1,Miss
3,1,1,female,35.0,1,0,53.1000,S,1,2,0,Mrs
4,0,3,male,35.0,0,0,8.0500,S,0,1,1,Mr


## 4. Data Transformation

- Encode categoricals: `Sex`, `Embarked`, `Title` (label-encoded here for
  the exploratory dataset export; the modeling pipeline in Section 6
  uses **OneHotEncoder** instead, which is the safer choice for
  non-ordinal categories and avoids implying a false ordering).
- Scale numeric features: `Age`, `Fare`, `FamilySize` using
  `StandardScaler` (again, done "for real" inside the pipeline in
  Section 6 to avoid leakage — the version here is for inspection only).
- Check for outliers (IQR method) in `Age` and `Fare`.


In [15]:
# 4.1 Outlier check (IQR method) - for reporting purposes
def iqr_outlier_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Age', 'Fare']:
    low, high = iqr_outlier_bounds(df_clean[col])
    n_out = ((df_clean[col] < low) | (df_clean[col] > high)).sum()
    print(f"{col}: bounds=({low:.2f}, {high:.2f}), outliers={n_out} ({n_out/len(df_clean)*100:.1f}%)")


Age: bounds=(2.50, 54.50), outliers=66 (7.4%)
Fare: bounds=(-26.72, 65.63), outliers=116 (13.0%)


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
sns.boxplot(y=df_clean['Age'], ax=axes[0], color='#1f77b4')
axes[0].set_title('Age (boxplot)')
sns.boxplot(y=df_clean['Fare'], ax=axes[1], color='#ff7f0e')
axes[1].set_title('Fare (boxplot)')
plt.tight_layout()
plt.savefig('outputs/outlier_boxplots.png', dpi=100)
plt.show()


In [17]:
# Fare has a long right tail with genuine (not erroneous) high values (first-class fares).
# Rather than deleting these real observations, we cap Fare at the 99th percentile
# to reduce the influence of extreme values while keeping the records.
fare_cap = df_clean['Fare'].quantile(0.99)
n_capped = (df_clean['Fare'] > fare_cap).sum()
df_clean['Fare'] = np.where(df_clean['Fare'] > fare_cap, fare_cap, df_clean['Fare'])
print(f"Capped {n_capped} Fare values at the 99th percentile ({fare_cap:.2f})")


Capped 9 Fare values at the 99th percentile (249.01)


In [18]:
# 4.2 Encode categoricals (label-encoding for the exported clean CSV / EDA copy)
from sklearn.preprocessing import LabelEncoder

df_encoded = df_clean.copy()
label_encoders = {}
for col in ['Sex', 'Embarked', 'Title']:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

for col, mapping in label_encoders.items():
    print(col, '->', mapping)


Sex -> {'female': np.int64(0), 'male': np.int64(1)}
Embarked -> {'C': np.int64(0), 'Q': np.int64(1), 'S': np.int64(2)}
Title -> {'Master': np.int64(0), 'Miss': np.int64(1), 'Mr': np.int64(2), 'Mrs': np.int64(3), 'Rare': np.int64(4)}


In [19]:
# 4.3 Scale numeric features (illustrative copy; the real, leak-safe scaling
# happens inside the Section 6 pipeline fit only on the training split)
from sklearn.preprocessing import StandardScaler

scale_cols = ['Age', 'Fare', 'FamilySize']
scaler_preview = StandardScaler()
df_scaled_preview = df_encoded.copy()
df_scaled_preview[scale_cols] = scaler_preview.fit_transform(df_scaled_preview[scale_cols])
df_scaled_preview.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,FamilySize,IsAlone,Title
0,0,3,1,-0.565736,1,0,-0.564109,2,0,0.059160,0,2
1,1,1,0,0.663861,1,0,0.942548,0,1,0.059160,0,3
2,1,3,0,-0.258337,0,0,-0.548227,2,0,-0.560975,1,1
3,1,1,0,0.433312,1,0,0.514708,2,1,0.059160,0,3
4,0,3,1,0.433312,0,0,-0.545285,2,0,-0.560975,1,2


## 5. Dataset Preparation

Split the **cleaned, feature-engineered but not-yet-encoded** dataframe
(`df_clean`) into `X` and `y`, then into train/test sets. Encoding and
scaling are deferred to the Section 6 pipeline so that they are fit
**only on the training data** — this avoids data leakage from the test
set into the imputation/scaling statistics.


In [20]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True).round(3))
print("Test target distribution:\n",  y_test.value_counts(normalize=True).round(3))


X_train: (712, 11)  X_test: (179, 11)
Train target distribution:
 Survived
0    0.617
1    0.383
Name: proportion, dtype: float64
Test target distribution:
 Survived
0    0.615
1    0.385
Name: proportion, dtype: float64


## 6. Preprocessing Pipeline (Scikit-learn)

A `ColumnTransformer` combining:
- **Numeric branch**: `SimpleImputer(strategy='median')` -> `StandardScaler`
  for `Age`, `Fare`, `SibSp`, `Parch`, `FamilySize`, `Pclass`.
- **Categorical branch**: `SimpleImputer(strategy='most_frequent')` ->
  `OneHotEncoder(handle_unknown='ignore')` for `Sex`, `Embarked`, `Title`.
- **Binary passthrough**: `IsAlone`, `HasCabin` (already 0/1, just imputed
  defensively).

The pipeline is **fit on `X_train` only** and then used to `transform`
both `X_train` and `X_test`, so no information from the test set leaks
into imputation medians / modes or scaling parameters.


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'Pclass']
categorical_features = ['Sex', 'Embarked', 'Title']
binary_features = ['IsAlone', 'HasCabin']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('bin', binary_transformer, binary_features)
])

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [22]:
# Fit ONLY on training data, then transform both splits (no leakage)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Recover feature names for a readable transformed dataframe
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = list(ohe.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_feature_names + binary_features

X_train_df = pd.DataFrame(X_train_transformed, columns=all_feature_names, index=X_train.index)
X_test_df  = pd.DataFrame(X_test_transformed,  columns=all_feature_names, index=X_test.index)

print("Transformed X_train shape:", X_train_df.shape)
X_train_df.head()


Transformed X_train shape: (712, 18)


,Age,Fare,SibSp,Parch,FamilySize,Pclass,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,IsAlone,HasCabin
692,-0.112078,0.606478,-0.465084,-0.466183,-0.556339,0.829568,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
481,-0.112078,-0.738619,-0.465084,-0.466183,-0.556339,-0.370945,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
527,-0.112078,4.541678,-0.465084,-0.466183,-0.556339,-1.571457,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
855,-0.879807,-0.516007,-0.465084,0.727782,0.073412,0.829568,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
801,0.118241,-0.113638,0.478335,0.727782,0.703162,-0.370945,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 7. Validation

In [23]:
# 7.1 No missing values remain
print("Missing values in transformed X_train:", int(X_train_df.isnull().sum().sum()))
print("Missing values in transformed X_test: ", int(X_test_df.isnull().sum().sum()))

# 7.2 All features are numerical
print("\nAll dtypes numeric (train):", all(np.issubdtype(dt, np.number) for dt in X_train_df.dtypes))
print("All dtypes numeric (test): ", all(np.issubdtype(dt, np.number) for dt in X_test_df.dtypes))

# 7.3 No data leakage: pipeline statistics were fit on X_train only (verified by construction above:
#     preprocessor.fit_transform(X_train) then preprocessor.transform(X_test) — transform never re-fits).
print("\nPipeline fit only on training data: True (fit_transform called on X_train, transform-only on X_test)")

# 7.4 Class distribution preserved by stratified split
orig_dist = y.value_counts(normalize=True).round(3)
train_dist = y_train.value_counts(normalize=True).round(3)
test_dist = y_test.value_counts(normalize=True).round(3)
print("\nClass distribution — full:\n", orig_dist)
print("Class distribution — train:\n", train_dist)
print("Class distribution — test:\n", test_dist)


Missing values in transformed X_train: 0
Missing values in transformed X_test:  0

All dtypes numeric (train): True
All dtypes numeric (test):  True

Pipeline fit only on training data: True (fit_transform called on X_train, transform-only on X_test)

Class distribution — full:
 Survived
0    0.616
1    0.384
Name: proportion, dtype: float64
Class distribution — train:
 Survived
0    0.617
1    0.383
Name: proportion, dtype: float64
Class distribution — test:
 Survived
0    0.615
1    0.385
Name: proportion, dtype: float64


## 8. Save Outputs

In [24]:
import joblib
import os

os.makedirs('outputs', exist_ok=True)

# Clean dataset (feature-engineered, pre-encoding — human-readable)
df_clean.to_csv('outputs/titanic_clean.csv', index=False)

# Also save the fully encoded/scaled illustrative version from Section 4
df_scaled_preview.to_csv('outputs/titanic_clean_encoded_scaled.csv', index=False)

# Train/test splits (raw, pre-pipeline — reproducible from these + the pipeline)
X_train.to_csv('outputs/X_train.csv', index=False)
X_test.to_csv('outputs/X_test.csv', index=False)
y_train.to_csv('outputs/y_train.csv', index=False)
y_test.to_csv('outputs/y_test.csv', index=False)

# Transformed (pipeline-output) train/test features, for direct model consumption
X_train_df.to_csv('outputs/X_train_transformed.csv', index=False)
X_test_df.to_csv('outputs/X_test_transformed.csv', index=False)

# Fitted preprocessing pipeline
joblib.dump(preprocessor, 'outputs/preprocessing_pipeline.joblib')

print("Saved all outputs to outputs/:")
for f in sorted(os.listdir('outputs')):
    print(' -', f)


Saved all outputs to outputs/:
 - X_test.csv
 - X_test_transformed.csv
 - X_train.csv
 - X_train_transformed.csv
 - outlier_boxplots.png
 - preprocessing_pipeline.joblib
 - target_distribution.png
 - titanic_clean.csv
 - titanic_clean_encoded_scaled.csv
 - y_test.csv
 - y_train.csv


## Summary

| Step | Decision |
|---|---|
| Duplicates | Dropped exact duplicate rows |
| Age missing | Imputed with median |
| Fare missing | Imputed with median |
| Embarked missing | Imputed with mode |
| Cabin (77% missing) | Converted to binary `HasCabin`, raw column dropped |
| PassengerId, Ticket | Dropped (identifiers, no predictive value) |
| Name | Used to extract `Title`, then dropped |
| New features | `FamilySize`, `IsAlone`, `Title`, `HasCabin` |
| Categorical encoding | OneHotEncoder (Sex, Embarked, Title) inside pipeline |
| Numeric scaling | StandardScaler (Age, Fare, SibSp, Parch, FamilySize, Pclass) inside pipeline |
| Outliers | Fare capped at 99th percentile; no rows deleted |
| Split | 80/20, `random_state=42`, stratified on `Survived` |
| Leakage prevention | `ColumnTransformer` fit on `X_train` only, applied to `X_test` via `.transform()` |

See `report.md` for the full write-up.
